## 2. Training Data Prep - Validation

This prepares a training dataset for a baseline of the "original method" of simple LR with minimal pre-processing, and no promotional calendar or discount schedules.

The results are grouped by Weekly date granularity, with Sundays starting the media week.

Our main predictor variable is Spend, and the response variable is Net Demand.

Use Last-Click Net Demand only.

### All Funnels

- Source: Tableau/Redshift BR Daily Actuals
- Metrics: Spend, LCND
- Channels:
    - All


In [1]:
import pandas as pd
import numpy as np

In [2]:
# notebook config
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

# packages
import pandas as pd

# functions
import functions.lr_models as lr
import functions.transform as tf

In [ ]:
# config
config = {
    'run': '202605_valid_baseline', # unique name of the run, e.g. '2025-01'
}

# 202603_1P
# 202603_2P
# 202603_3P
# 202603_reflows
# 202604_dev
# 202604_mmm
# 202604_1P
# 202604_2P
# 202604_3P
# 202605_1P
# 202605_2P

## All Funnels
----

In [4]:
# load data exported from Tableau as a CSV format, make sure this is the name of the file: BR Pacing Actuals.csv
# handling for tab-separated values

data_lf = tf.clean_df(
    pd.read_csv(
        '../data/raw/BR Pacing Actuals_20260502.csv',
        sep='\t',
        encoding='utf-16'))

# handling for comma-separated values

# data_lf = tf.clean_df(
#     pd.read_csv(
#         '../data/raw/BR Pacing Actuals_20260218.csv',
#         encoding='utf-8'))

print(data_lf.shape)

(29111, 17)


In [5]:
# columns
# standardize names
data_lf = data_lf[['date', 'brand', 'channel', 'funnel', 'tactic', 'spend', 'lcnd']]
cols = ['date', 'brand', 'channel', 'funnel', 'tactic', 'spend', 'nd']
data_lf.columns = cols

In [6]:
# dates 

# date in datetime format
data_lf['date'] = pd.to_datetime(data_lf['date'])

# calculate weekstart and fiscal month
data_lf['weekstart'] = data_lf['date'] - pd.to_timedelta((data_lf['date'].dt.dayofweek + 1) % 7, unit='D')

In [7]:
# brand
print(data_lf['brand'].unique())

# standardize names
data_lf['brand'] = data_lf['brand'].apply(
    lambda x: 'Factory US' if 'factory' in x.lower() else 
              'Specialty CA' if 'canada' in x.lower() else 
              'Specialty US')

# qa
print(data_lf['brand'].unique())

['brand us' 'Brand US' 'brand ca'
 'brand outlet' 'Brand CA'
 'Brand Outlet']
['Specialty US' 'Specialty CA' 'Factory US']


In [8]:
# funnel
data_lf['funnel'] = data_lf['funnel'].map({
    'Awareness': 'AWE',
    'Consideration': 'CON',
    'Conversion': 'CVR'
})

# qa
print(data_lf['funnel'].unique())

['CVR' 'AWE' 'CON']


In [9]:
# platform

data_lf['compare'] = data_lf['channel'] + ' ' + data_lf['funnel'] + ' ' + data_lf['tactic']

# extract or infer platform from tactic or channel
conditions = [
    data_lf['tactic'].str.contains('Ad Marketplace', case=False, na=False),
    data_lf['tactic'].str.contains('Pinterest', case=False, na=False),
    data_lf['tactic'].str.contains('Meta', case=False, na=False),
    data_lf['tactic'].str.contains('YouTube', case=False, na=False),
    data_lf['tactic'].str.contains('TikTok', case=False, na=False),
    data_lf['tactic'].str.contains('CVR ASC', case=False, na=False),
    data_lf['channel'].str.lower() == 'search',
    data_lf['channel'].str.lower() == 'affiliates',
    data_lf['tactic'].str.contains('CVR', case=False, na=False),
]

choices = ['Ad Marketplace', 'Pinterest', 'Meta', 'YouTube', 'TikTok','Meta', 'Google', 'NA', 'Meta']

data_lf['platform'] = np.select(conditions, choices, default='NA')

# qa
df_schema = data_lf[['compare', 'channel','funnel','platform','tactic']].drop_duplicates().sort_values(by=['channel','funnel','platform','tactic'])

print(data_lf['platform'].unique())

['NA' 'Google' 'Meta' 'Pinterest' 'TikTok' 'YouTube' 'Ad Marketplace']


In [10]:
# tactic

data_lf['tactic'] = data_lf['tactic'].str.replace(r'\b(Meta|Pinterest|TikTok|YouTube|AWE|CON|CVR|Aff)\b', '', regex=True).str.strip()
data_lf['tactic'] = data_lf['tactic'].str.replace(r'\(BAU\)', '', regex=True).str.strip()
data_lf['tactic'] = data_lf['tactic'].replace(['Ad Marketplace', 'Not Applicable'], 'NA')
data_lf['tactic'] = data_lf['tactic'].replace('Non Brand', 'NonBrand')
data_lf['tactic'] = data_lf['tactic'].str.replace(r'\b(Test|Credits|Credit)$', '', regex=True) # aggregating 'Test' and 'Credit' versions
data_lf['tactic'] = data_lf['tactic'].replace(r'^\s*$', 'NA', regex=True)
data_lf['tactic'] = data_lf['tactic'].str.strip()

# qa
df_schema = data_lf[['compare', 'channel','funnel','platform','tactic']].drop_duplicates().sort_values(by=['channel','funnel','platform','tactic'])

print(data_lf['tactic'].unique())

['NA' 'Brand' 'DAB' 'DPA' 'NonBrand' 'PLA' 'ASC' 'CUS' 'PMAX' 'RSC'
 'Shorts' 'DYN' 'Content' 'Coupon' 'Loyalty' 'Search' 'ASC Volume'
 'ASC Omni' 'ASC Value' 'Acquire' 'Retention' 'Acquire & Reclaim' 'MAX'
 'Stores' 'View Content']


In [11]:
# # channel and funnel
# # filter rows to keep lower funnel only
# data_lf = data_lf[
#     (data_lf['channel'].isin(['Affiliate', 'Search'])) |
#     ((data_lf['channel'] == 'Social') & (data_lf['funnel'] == 'CVR'))]

# # qa
# df_schema = data_lf[['compare', 'brand', 'channel','funnel','platform','tactic']].drop_duplicates().sort_values(by=['brand', 'channel','funnel','platform','tactic'])

In [12]:
# recreate 'channel' columm, clean and order
data_lf['channel'] = data_lf['channel'] + ' ' + data_lf['funnel'] + ' ' + data_lf['platform'] + ' ' + data_lf['tactic']

data_lf = data_lf[['date','weekstart','brand','channel','spend','nd']].sort_values(by=['brand','channel','date','weekstart'])

# qa
data_lf['channel'].unique()

array(['Affiliates CON NA Content', 'Affiliates CVR NA Coupon',
       'Affiliates CVR NA Loyalty', 'Affiliates CVR NA NA',
       'Search CVR Google Brand', 'Search CVR Google NonBrand',
       'Search CVR Google PLA', 'Search CVR Google PMAX',
       'Search CVR Google RSC', 'Social AWE Meta NA',
       'Social CON Meta NA', 'Social CON NA NA', 'Social CVR Meta ASC',
       'Social CVR Meta ASC Omni', 'Social CVR Meta ASC Value',
       'Social CVR Meta ASC Volume', 'Social CVR Meta CUS',
       'Social CVR Meta DAB', 'Social CVR Meta DPA',
       'Social AWE Pinterest NA', 'Social CON Pinterest DYN',
       'Search CON Google NonBrand', 'Search CVR Ad Marketplace NA',
       'Social AWE Meta Acquire', 'Social AWE Meta Retention',
       'Social AWE Meta Stores', 'Social AWE TikTok NA',
       'Social AWE YouTube Shorts', 'Social CON Meta Acquire & Reclaim',
       'Social CON Meta Retention', 'Social CON Pinterest CUS',
       'Social CON Pinterest MAX', 'Social CON Pinterest Search

In [13]:
# numeric values
# spend and nd; remove currency, handle parentheses for negative values, and convert to decimal
data_lf['spend'] = data_lf['spend'].replace('[\$,]', '', regex=True).replace(r'\((.*)\)', r'-\1', regex=True).fillna(0).astype(float)
data_lf['nd'] = data_lf['nd'].replace('[\$,]', '', regex=True).replace(r'\((.*)\)', r'-\1', regex=True).fillna(0).astype(float).round(4)

In [14]:
# date grouping
calendar = tf.clean_df(pd.read_csv(f"../data/meta/fiscal_calendar.csv"))
calendar['weekstart'] = pd.to_datetime(calendar['weekstart'], format='%m/%d/%y')

# merge in fyear and fmonth
data_lf = data_lf.merge(calendar[['weekstart','fyear', 'fmonth']], on='weekstart', how='left')

In [ ]:
# # filter out custom ranges or conditions as needed
# # example: filter out Fiscal November ("peak")
# # data_lf = data_lf[data_lf['fmonth'] != 10] 

# # remove BRSP Search CVR NonBrand, greater stability after this point.
# data_lf = data_lf[~(
#     (data_lf['brand'] == 'Specialty US') & 
#     (data_lf['channel'] == 'Search CVR Google NonBrand') & 
#     (data_lf['weekstart'] < '2025-07-06')
# )]

In [15]:
# group by week
data_lf_weekly = data_lf.groupby(['brand', 'channel', 'weekstart'], as_index=False)[['spend', 'nd']].sum()

# be sure to manually QA
data_lf_weekly.to_csv(f"../data/qa/data_lf_{config['run']}.csv")
data_lf_weekly.head(5)

,brand,channel,weekstart,spend,nd
0,Factory US,Affiliates CON NA Content,2025-05-04,8717.14,105147.46
1,Factory US,Affiliates CON NA Content,2025-05-11,15101.25,218612.07
2,Factory US,Affiliates CON NA Content,2025-05-18,13345.30,154020.11
3,Factory US,Affiliates CON NA Content,2025-05-25,13883.34,160668.31
4,Factory US,Affiliates CON NA Content,2025-06-01,6996.60,91906.08


### Final Output

In [17]:
data_lf_weekly.to_csv(f"../data/train/train_{config['run']}.csv", index=False)

# QA
data_lf_weekly.head(5)

,brand,channel,weekstart,spend,nd
0,Factory US,Affiliates CON NA Content,2025-05-04,8717.14,105147.46
1,Factory US,Affiliates CON NA Content,2025-05-11,15101.25,218612.07
2,Factory US,Affiliates CON NA Content,2025-05-18,13345.30,154020.11
3,Factory US,Affiliates CON NA Content,2025-05-25,13883.34,160668.31
4,Factory US,Affiliates CON NA Content,2025-06-01,6996.60,91906.08
